# 03 — Concentration Error Sensitivity

How sensitive is $K_D$ to errors in the injected analyte concentration?

LCMS-measured concentrations have inherent uncertainty. If the nominal
concentration is wrong, the fitted $k_a$ (and therefore $K_D = k_d/k_a$)
shifts because $c(t)$ is scaled incorrectly.

**Protocol:** For a single sample, perturb the nominal concentration by
±10%, ±30%, ±60%, ±90% and re-fit with ODE.

$k_d$ is independent of $c(t)$ (dissociation-only), so only $k_a$ and $R_{max}$ change.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sensofit.data_loader import load_cxw
from sensofit.models import (
    build_concentration_profile, build_pulsed_concentration_profile,
    select_dmso_cal, select_blank,
    double_reference, build_weight_mask, build_full_weight_mask,
    smooth_and_differentiate, is_nonspecific_binder
)
from sensofit.ode_fitting import ode_fit, fit_sample as ode_fit_sample

CXW = '../20250826_DENV-2 NS2B3 Binding Assay.cxw'
data = load_cxw(CXW)
samples = data['samples']
dmso_cals = data['dmso_cals']
blanks = data['blanks']

def find_sample(name, conc_uM=None):
    """Find a sample by compound name (substring match), optionally filter by concentration.
    If multiple matches, prompts user to choose."""
    hits = [(i, s) for i, s in enumerate(samples) if name in s['compound']]
    if conc_uM is not None:
        hits = [(i, s) for i, s in hits if abs(s['concentration_M'] * 1e6 - conc_uM) < 0.01]
    if not hits:
        raise ValueError(f'No sample found matching "{name}"' + (f' at {conc_uM} µM' if conc_uM else ''))
    if len(hits) == 1:
        return hits[0][1]
    # Multiple matches — ask user to pick
    print(f'Multiple matches for "{name}":')
    for j, (idx, s) in enumerate(hits):
        nsb, _ = is_nonspecific_binder(s)
        tag = ' [NSB]' if nsb else ''
        print(f'  [{j}] samples[{idx}]  {s["compound"]}  {s["concentration_M"]*1e6:.0f} µM  (cycle {s["index"]}){tag}')
    choice = int(input(f'Select [0–{len(hits)-1}]: '))
    return hits[choice][1]

# --- Select sample here ---
sample = find_sample('ASAP-0044216')
# sample = samples[30]  # or pick by index
print(f'Sample: {sample["compound"]}  ({sample["concentration_M"]*1e6:.0f} µM, cycle {sample["index"]})')

## 1. Baseline Fit (true concentration)

In [ ]:
ode_true = ode_fit_sample(sample, dmso_cals, blanks=blanks)
print(f'ODE baseline:  ka={ode_true["ka"]:.2e},  kd={ode_true["kd"]:.4f},  '
      f'Rmax={ode_true["Rmax"]:.1f},  KD={ode_true["KD"]*1e6:.2f} µM')

## 2. Perturbed Concentration Sweep

For each perturbation level, rebuild $c(t)$ with the perturbed concentration and refit.

In [ ]:
perturbations = [-0.9, -0.6, -0.3, -0.1, 0.0, 0.1, 0.3, 0.6, 0.9]

C_true = sample['concentration_M']

results = []

for delta in perturbations:
    C_pert = C_true * (1.0 + delta)

    # Temporarily patch concentration and fit via full ODE pipeline
    sample_pert = {**sample, 'concentration_M': C_pert}
    try:
        ode_res = ode_fit_sample(sample_pert, dmso_cals, blanks=blanks)
        ka = ode_res['ka']
        kd = ode_res['kd']
        Rmax = ode_res['Rmax']
        KD = ode_res['KD']
        success = ode_res['success']
    except Exception:
        ka = kd = Rmax = KD = np.nan
        success = False

    results.append({
        'delta_pct': delta * 100,
        'C_uM': C_pert * 1e6,
        'ka': ka, 'kd': kd, 'Rmax': Rmax,
        'KD_uM': KD * 1e6 if not np.isnan(KD) else np.nan,
        'success': success,
    })

df = pd.DataFrame(results)
print(df[['delta_pct', 'C_uM', 'ka', 'kd', 'Rmax', 'KD_uM', 'success']].to_string(index=False))

## 3. KD Sensitivity Plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Show which fits failed
failed = df[~df['success']]
if len(failed):
    print(f'Warning: ODE fit failed at perturbations: {failed["delta_pct"].tolist()}%')

# KD vs concentration error
ax = axes[0]
ax.plot(df['delta_pct'], df['KD_uM'], 'rs-')
ax.axhline(df.loc[df['delta_pct'] == 0, 'KD_uM'].values[0],
           color='red', ls=':', alpha=0.5)
ax.set_xlabel('Concentration error (%)')
ax.set_ylabel('KD (µM)')
ax.set_title('KD sensitivity to [analyte] error')
ax.set_xlim(-100, 100)
ax.grid(True, alpha=0.3)

# ka vs concentration error
ax = axes[1]
ax.plot(df['delta_pct'], df['ka'], 'rs-')
ax.set_xlabel('Concentration error (%)')
ax.set_ylabel('ka (M⁻¹s⁻¹)')
ax.set_title('ka sensitivity')
ax.set_xlim(-100, 100)
ax.grid(True, alpha=0.3)

# kd should be invariant
ax = axes[2]
ax.plot(df['delta_pct'], df['kd'], 'rs-')
ax.set_xlabel('Concentration error (%)')
ax.set_ylabel('kd (s⁻¹)')
ax.set_title('kd (should be invariant)')
ax.set_xlim(-100, 100)
ax.grid(True, alpha=0.3)

plt.suptitle(f'{sample["compound"]} — true conc = {C_true*1e6:.0f} µM',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 4. Relative Error Table

In [ ]:
# Compute relative errors vs true (delta=0)
true_row = df[df['delta_pct'] == 0].iloc[0]

df['KD_rel_err'] = (df['KD_uM'] - true_row['KD_uM']) / true_row['KD_uM'] * 100
df['ka_rel_err'] = (df['ka'] - true_row['ka']) / true_row['ka'] * 100
df['kd_rel_err'] = (df['kd'] - true_row['kd']) / true_row['kd'] * 100

print('Relative error (%) in kinetic parameters vs true concentration:')
print(df[['delta_pct', 'KD_rel_err', 'ka_rel_err', 'kd_rel_err']].to_string(index=False))

In [ ]:
# Bar chart of KD relative error
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(df))
ax.bar(x, df['KD_rel_err'], color='indianred')
ax.set_xticks(x)
ax.set_xticklabels([f'{d:+.0f}%' for d in df['delta_pct']])
ax.set_xlabel('Concentration perturbation')
ax.set_ylabel('KD relative error (%)')
ax.set_title('KD error propagation from concentration uncertainty (ODE)')
ax.axhline(0, color='k', lw=0.5)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Log-ratio of KD: log10(KD_perturbed / KD_true)
KD_true = true_row['KD_uM']
df['log_KD_ratio'] = np.log10(df['KD_uM'] / KD_true)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df['delta_pct'], df['log_KD_ratio'], 'rs-', markersize=8)
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('Concentration error (%)')
ax.set_ylabel(r'$\log_{10}(K_D^{pert} \,/\, K_D^{true})$')
ax.set_title('Log-scale KD shift from concentration error (ODE)')
ax.set_xlim(-100, 100)
ax.grid(True, alpha=0.3)

# Add reference lines at ±0.5 log unit (≈3-fold error)
ax.axhline(0.5, color='gray', ls='--', alpha=0.4, label='±0.5 log (3×)')
ax.axhline(-0.5, color='gray', ls='--', alpha=0.4)
ax.legend()
plt.tight_layout()
plt.show()

## 5. Summary

Key takeaways:
- $k_d$ is **invariant** to concentration errors (fitted from dissociation only, where $c=0$)
- $k_a$ scales approximately inversely with concentration: $k_a \propto 1/c$
- $K_D = k_d / k_a$ therefore scales roughly linearly with concentration error
- At ±30% concentration error, expect ~30% KD error
- At ±90% concentration error, KD can be off by an order of magnitude